# Experiment 5 - Part 2: Semantic Search with SentenceTransformers

**Course:** HAAI302COM - Artificial Intelligence Laboratory

In Part 1, TF-IDF failed whenever a claim and its answer shared no words. A **sentence embedding model** turns any text into a fixed-length vector so that texts with similar *meaning* get nearby vectors, even with different words. Search then becomes: embed the query, find the closest document vectors.

We use the same SciFact benchmark and the same metrics so the comparison with TF-IDF is fair.

## 0. Setup

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

import ir_utils
data = ir_utils.load_scifact()
doc_texts = [t + ". " + x for t, x in zip(data["titles"], data["texts"])]
print(len(doc_texts), "documents |", len(data["q_texts"]), "queries")

5183 documents | 300 queries


For comparison we recompute the TF-IDF results from Part 1, using its best setting (`sublinear_tf`, English stop words, 1-2 word n-grams) and the plain default.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def tfidf_run(**kw):
    v = TfidfVectorizer(**kw)
    X = v.fit_transform(doc_texts)
    S = linear_kernel(v.transform(data["q_texts"]), X)
    return np.argsort(-S, axis=1)[:, :100]

tfidf_default = tfidf_run()
tfidf_best = tfidf_run(sublinear_tf=True, stop_words="english", ngram_range=(1, 2))
base = pd.DataFrame({"tfidf default": ir_utils.evaluate(tfidf_default, data["relevant"]),
                     "tfidf best": ir_utils.evaluate(tfidf_best, data["relevant"])}).T
base

,recall@1,recall@10,recall@100,mrr@10,ndcg@10
tfidf default,0.4301,0.7135,0.8836,0.5437,0.5780
tfidf best,0.4850,0.7790,0.8916,0.6047,0.6418


## 1. The embedding model

`all-MiniLM-L6-v2` is a small (22 million parameters) and fast model trained on over a billion sentence pairs so that sentences with the same meaning get high cosine similarity. A `SentenceTransformer` is a pipeline of modules; printing it shows each stage.

In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(model)
print("embedding size:", model.get_sentence_embedding_dimension(), "| max input tokens:", model.max_seq_length)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
embedding size: 384 | max input tokens: 256


**What to notice:** three stages. A Transformer (the language model, 6 layers) turns each token into a vector; `Pooling` averages the token vectors into one vector per text; `Normalize` scales that vector to length 1. Because of the last stage, a dot product of two embeddings is their cosine similarity. And note `max_seq_length = 256`: **longer inputs are cut off**. Our abstracts average 200 words, so how many exceed that limit?

In [4]:
tok = model.tokenizer
n_tokens = np.array([len(tok(t, add_special_tokens=True, verbose=False)["input_ids"]) for t in doc_texts[:1000]])
print("tokens per abstract (first 1000): mean", int(n_tokens.mean()), "| median", int(np.median(n_tokens)), "| max", n_tokens.max())
print("share longer than 256 tokens (truncated):", round(float((n_tokens > 256).mean()), 2))

tokens per abstract (first 1000): mean 332 | median 308 | max 1475
share longer than 256 tokens (truncated): 0.71


**What to notice:** the median abstract is 308 tokens and 71 percent of them are longer than the 256-token limit, so for most documents the model never sees the end of the text. Words are split into sub-word tokens, so 200 words become about 330 tokens. We test later how much this truncation matters.

## 2. Embeddings of a few sentences

Before running on the corpus, a tiny experiment to build intuition: four sentences, two pairs with the same meaning but different words.

In [5]:
sents = ["Tamoxifen reduces the risk of breast cancer recurrence.",
         "A drug lowers the chance that mammary tumours come back.",
         "The new smartphone has a very bright display.",
         "This phone's screen is extremely luminous."]
E = model.encode(sents)
print("shape:", E.shape, "| norms:", np.linalg.norm(E, axis=1).round(3))
pd.DataFrame(model.similarity(E, E).numpy().round(2))

shape: (4, 384) | norms: [1. 1. 1. 1.]


,0,1,2,3
0,1.00,0.56,0.03,-0.01
1,0.56,1.00,0.07,0.02
2,0.03,0.07,1.00,0.68
3,-0.01,0.02,0.68,1.00


**What to notice:** sentences 0 and 1 share almost no words (only "the") yet score 0.56, and the phone pair scores 0.68; across topics the scores are close to zero (0.03, -0.01, 0.07, 0.02). TF-IDF would give 0 for 0 vs 1, since no important word matches. The matrix is symmetric with 1.0 on the diagonal, as expected for normalised vectors.

## 3. Encoding the corpus

This is the expensive step: 5,183 abstracts through a transformer. On a CPU it takes several minutes, so we save the result to a `.npy` file and reuse it. On a GPU (Colab) it takes seconds.

In [6]:
def encode_cached(texts, path, **kw):
    if os.path.exists(path):
        return np.load(path)
    t0 = time.time()
    E = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False, **kw)
    np.save(path, E)
    print(f"encoded {len(texts)} texts in {time.time() - t0:.0f} s")
    return E

D = encode_cached(doc_texts, "emb_minilm_title_text.npy")
print("document matrix:", D.shape, D.dtype, "| size:", round(D.nbytes / 1e6, 1), "MB")

document matrix: (5183, 384) float32 | size: 8.0 MB


In [7]:
t0 = time.time()
Q = model.encode(data["q_texts"], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
print("query matrix:", Q.shape, "| encoded in", round(time.time() - t0, 1), "s")

query matrix: (300, 384) | encoded in 2.5 s


**What to notice:** the corpus becomes a dense 5,183 x 384 matrix of 32-bit floats (8 MB; compare with the sparse TF-IDF matrix of 36,000 columns). Every number is non-zero. Encoding the 5,183 abstracts took 281 seconds on this CPU while the 300 short queries took 2.5 seconds: the long documents are what cost time, and it is a one-time cost that we cache.

## 4. Searching: one matrix multiplication

Because the vectors are normalised, the cosine similarity between every query and every document is a single matrix product `Q @ D.T`. Sorting each row gives the ranking.

In [8]:
scores = Q @ D.T                                   # (300 queries, 5183 documents) of cosine similarities
dense = np.argsort(-scores, axis=1)[:, :100]
print("score matrix:", scores.shape, "| score range:", scores.min().round(2), "to", scores.max().round(2))

i = 3
print("\nCLAIM:", data["q_texts"][i])
pd.DataFrame({"rank": range(1, 6), "cosine": scores[i, dense[i, :5]].round(3),
              "relevant": [d in data["relevant"][i] for d in dense[i, :5]],
              "title": [data["titles"][d][:80] for d in dense[i, :5]]})

score matrix: (300, 5183) | score range: -0.27 to 0.88

CLAIM: 5% of perinatal mortality is due to low birth weight.


,rank,cosine,relevant,title
0,1,0.523,False,Perinatal mortality in rural China: retrospective cohort study.
1,2,0.508,False,Neonatal Mortality Levels for 193 Countries in 2009 with Trends since 1990: A Sy
2,3,0.486,False,Maternal and child undernutrition: consequences for adult health and human capit
3,4,0.484,False,Determinants of neonatal mortality in Indonesia
4,5,0.483,False,Evidence-based interventions for improvement of maternal and child nutrition: wh


**What to notice:** this is the same claim as the TF-IDF example in Part 1 (perinatal mortality and low birth weight). The dense ranking returns a different top five: neonatal mortality levels, determinants of neonatal mortality, child undernutrition. They are closer in topic than the TF-IDF results (which included breast cancer papers), yet none of them is the judged-relevant abstract about underweight prevalence. Semantic search found the right *topic*, not the right *document*, for this claim. The scores are cosines of 0.48 to 0.52.

For production-style use, the library has this built in: `util.semantic_search` returns the top hits per query. We confirm it agrees with our matrix product.

In [9]:
from sentence_transformers import util
hits = util.semantic_search(Q, D, top_k=10)                        # list (per query) of [{corpus_id, score}, ...]
same = np.mean([h[0]["corpus_id"] == dense[i, 0] for i, h in enumerate(hits)])
print("top-1 agrees with the manual ranking for", f"{same:.0%}", "of queries")

top-1 agrees with the manual ranking for

 100% of queries


## 5. Dense versus keyword: evaluation

In [10]:
dense_m, dense_q = ir_utils.evaluate(dense, data["relevant"], per_query=True)
cmp = pd.concat([base, pd.DataFrame({"dense MiniLM": dense_m}).T])
cmp

,recall@1,recall@10,recall@100,mrr@10,ndcg@10
tfidf default,0.4301,0.7135,0.8836,0.5437,0.5780
tfidf best,0.4850,0.7790,0.8916,0.6047,0.6418
dense MiniLM,0.4846,0.7883,0.9250,0.6068,0.6484


**What to notice:** against the default TF-IDF, dense search is clearly better (nDCG@10 0.648 vs 0.578, recall@100 0.925 vs 0.884). But against the **tuned** TF-IDF it is almost a tie: nDCG@10 0.648 vs 0.642 and recall@1 equal at 0.485. The main edge of dense search is deeper in the list: recall@100 rises from 0.892 to 0.925. Two lessons: always compare against a tuned baseline (Part 1 showed `sublinear_tf` alone is worth 0.05 nDCG), and do not expect embeddings to win everywhere. SciFact is full of precise scientific terms, which suits keyword matching.

The averages hide which queries each method wins. We compare them query by query on `recall@10`.

In [11]:
_, tfidf_q = ir_utils.evaluate(tfidf_best, data["relevant"], per_query=True)
diff = dense_q["recall@10"] - tfidf_q["recall@10"]
outcome = pd.cut(diff, [-2, -0.01, 0.01, 2], labels=["tfidf better", "tie", "dense better"])
print(outcome.value_counts().to_string())
print("\nqueries neither method finds in the top 10:", int(((dense_q["recall@10"] == 0) & (tfidf_q["recall@10"] == 0)).sum()))

recall@10
tie             234
dense better     37
tfidf better     29

queries neither method finds in the top 10: 33


**What to notice:** on 234 of 300 queries the two methods tie on recall@10. Dense search is better on 37 and TF-IDF on 29, and 33 queries are found by neither. The two methods fail on *different* queries, which is the motivation for combining them in notebook 03.

Examples of each kind make the difference concrete. First, claims where dense search finds the abstract and TF-IDF does not.

In [12]:
def show(idx, n=3):
    for i in idx[:n]:
        d = next(iter(data["relevant"][i]))
        print(f"CLAIM   : {data['q_texts'][i]}\nRELEVANT: {data['titles'][d][:110]}\n")

show(list(diff[(dense_q["recall@10"] > 0) & (tfidf_q["recall@10"] == 0)].index))

CLAIM   : 0-dimensional biomaterials show inductive properties.
RELEVANT: New opportunities: the use of nanotechnologies to manipulate and track stem cells.

CLAIM   : AMP-activated protein kinase (AMPK) activation increases inflammation-related fibrosis in the lungs.
RELEVANT: Metformin reverses established lung fibrosis in a bleomycin model

CLAIM   : Bone marrow cells contribute to adult macrophage compartments.
RELEVANT: Tissue-resident macrophages self-maintain locally throughout adult life with minimal contribution from circula



Now claims where TF-IDF wins and dense search misses.

In [13]:
show(list(diff[(dense_q["recall@10"] == 0) & (tfidf_q["recall@10"] > 0)].index))

CLAIM   : A total of 1,000 people in the UK are asymptomatic carriers of vCJD infection.
RELEVANT: Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large s

CLAIM   : Activation of PPM1D suppresses p53 function.
RELEVANT: Mosaic PPM1D mutations are associated with predisposition to breast and ovarian cancer

CLAIM   : Arterioles have a larger lumen diameter than venules.
RELEVANT: Scaffold-based three-dimensional human fibroblast culture provides a structural matrix that supports angiogene



**What to notice:** dense search wins when a concept is linked without shared words: "0-dimensional biomaterials" to "nanotechnologies", and "AMPK activation" to a paper on metformin (a drug that activates AMPK). TF-IDF wins on rare exact identifiers: the gene name `PPM1D` appears literally in the relevant title, and an acronym like vCJD matches the text while the embedding does not connect it to the prion abstract. Embedding models are weak on rare tokens they saw little in training. The third TF-IDF win ("arterioles have a larger lumen") has no obvious link to its relevant title, a reminder that some judgments are hard for both methods.

## 6. Experiment: what do we embed, and how much of it?

Two choices change the embedding quality: the text we feed in, and the length limit. We test two variants (the cache avoids re-encoding when you re-run).

- **Title only:** short and fast, loses the abstract.
- **`max_seq_length = 128`:** the model sees only the first 128 tokens of each abstract (about 90 words).

In [14]:
D_title = encode_cached(data["titles"], "emb_minilm_title_only.npy")

model.max_seq_length = 128
D_128 = encode_cached(doc_texts, "emb_minilm_128tok.npy")
model.max_seq_length = 256                              # restore the default

rows = {}
for name, M in {"title + abstract, 256 tokens (default)": D, "title + abstract, 128 tokens": D_128, "title only": D_title}.items():
    r = np.argsort(-(Q @ M.T), axis=1)[:, :100]
    rows[name] = ir_utils.evaluate(r, data["relevant"])
pd.DataFrame(rows).T

,recall@1,recall@10,recall@100,mrr@10,ndcg@10
"title + abstract, 256 tokens (default)",0.4846,0.7883,0.9250,0.6068,0.6484
"title + abstract, 128 tokens",0.4696,0.7620,0.9193,0.5828,0.6222
title only,0.4283,0.6904,0.8660,0.5323,0.5644


**What to notice:** reading the table from top to bottom, quality falls as the model sees less text. Cutting the abstract to the first 128 tokens costs 2.6 points of nDCG@10 (0.648 to 0.622) and encoding titles only costs 8.4 points (0.564). Titles are much cheaper (39 seconds against 281) but lose much information. The default 256-token run is better than 128 although 71 percent of abstracts are still truncated, so a model with a longer input limit, or splitting abstracts into chunks that are embedded separately (as in Experiment 4), is the next improvement to try.

## 7. What does "close" look like? Score distributions

A good embedding places the relevant document much closer to the query than random documents. We compare the cosine similarity of each query to its relevant abstract versus to all other abstracts.

In [15]:
rel_scores = np.concatenate([scores[i, list(r)] for i, r in enumerate(data["relevant"])])
mask = np.ones_like(scores, dtype=bool)
for i, r in enumerate(data["relevant"]):
    mask[i, list(r)] = False
other = scores[mask]
pd.DataFrame({"relevant pairs": pd.Series(rel_scores).describe(), "all other pairs": pd.Series(other).describe()}).round(3)

,relevant pairs,all other pairs
count,339.000,1554561.000
mean,0.563,0.104
std,0.149,0.112
min,0.085,-0.275
25%,0.459,0.022
50%,0.590,0.097
75%,0.668,0.178
max,0.882,0.842


**What to notice:** relevant pairs have a mean cosine of 0.56 (median 0.59) while the 1.5 million other pairs average 0.10, so the embedding separates them well on average. But the tails overlap: the lowest relevant pair scores 0.09, while the highest non-relevant pair scores 0.84. A good ranker only needs the relevant document to beat its competitors for the *same* query, which is why ranking metrics (and not a fixed score cutoff) are the right way to judge search.

## 8. Free-form semantic search

Finally, the engine in use: type any question and read the top results. Try your own queries by editing the string.

In [16]:
def search(text, k=5):
    q = model.encode([text], normalize_embeddings=True)[0]
    s = D @ q
    top = np.argsort(-s)[:k]
    return pd.DataFrame({"cosine": s[top].round(3), "title": [data["titles"][i][:95] for i in top]})

search("Does vitamin D supplementation prevent bone fractures?")

,cosine,title
0,0.670,Vitamin d deficiency in postmenopausal women - biological correlates.
1,0.648,Estimation of optimal serum concentrations of 25-hydroxyvitamin D for multiple health ...
2,0.611,Vitamin D and obesity: current perspectives and future directions.
3,0.609,"Relationship between serum parathyroid hormone levels, vitamin D sufficiency, and calc..."
4,0.600,Vitamin D insufficiency and the blunted PTH response in established osteoporosis: the ...


In [17]:
search("heart disease risk from eating a lot of red meat")

,cosine,title
0,0.574,Risk of hospitalization or death from ischemic heart disease among British vegetarians...
1,0.523,"Intestinal microbiota metabolism of L-carnitine, a nutrient in red meat, promotes athe..."
2,0.499,Proteomic approach to characterize biochemistry of meat quality defects.
3,0.498,The relation between dietary flavonol intake and coronary heart disease mortality: a m...
4,0.486,Vegetarian dietary patterns and mortality in Adventist Health Study 2.


**What to notice:** the vitamin D query returns five vitamin D papers, with cosines from 0.60 to 0.67. The red meat query is more mixed: ranks 1 and 2 are about diet and heart disease, but rank 3 is about meat quality defects in food processing and rank 4 about flavonols, with scores (0.50 and 0.50) barely lower than the useful rank 2 (0.52). So a similarity score is not a reliable "is this relevant" signal across queries; it only orders results for the same query. A confidence threshold has to be calibrated on your own data.

## Summary

| Idea | What we saw |
|---|---|
| Embedding model | Transformer + pooling + normalise; 384 numbers per text; dot product = cosine |
| Truncation | the model only reads 256 tokens; long abstracts are cut |
| Search | one matrix product `Q @ D.T`, then sort; `util.semantic_search` does the same |
| Versus TF-IDF | see the table in section 5: dense search wins on average but not on every query |
| Cost | encoding the corpus is the expensive step, so cache it |

**Next:** Notebook 03 replaces the brute-force matrix product with FAISS indexes, trades accuracy for speed and memory, and fuses dense and keyword search.